In [3]:
import pandas as pd
import numpy as np

# Load dữ liệu
train_df = pd.read_csv("../04_DataSet/train.csv")
val_df = pd.read_csv("../04_DataSet/dev.csv")
test_df = pd.read_csv("../04_DataSet/test.csv")

# Chuẩn hóa tên cột đồng bộ
col_map = {
    'rating': 'Rating',
    'comment': 'Comment',
    'label': 'Label',
    'spam_label': 'SpamLabel'
}
train_df = train_df.rename(columns=col_map)
val_df = val_df.rename(columns=col_map)
test_df = test_df.rename(columns=col_map)

print(f"Kích thước Train: {train_df.shape}, Dev: {val_df.shape}, Test: {test_df.shape}")


Kích thước Train: (14306, 11), Dev: (1590, 11), Test: (3974, 11)


In [4]:
# Xử lý giá trị bị thiếu, và ép kiểu dữ liệu
for df in [train_df, val_df, test_df]:
    df['Comment'] = df['Comment'].fillna("").astype(str)
    if 'category' in df.columns:
        df['category'] = df['category'].fillna("Khac").astype(str)
    if 'num_sold' in df.columns:
        df['num_sold'] = pd.to_numeric(df['num_sold'], errors='coerce').fillna(0)
    if 'num_review' in df.columns:
        df['num_review'] = pd.to_numeric(df['num_review'], errors='coerce').fillna(0)
    if 'Rating' in df.columns:
        df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce').fillna(5.0)
print("Da xu ly gia thieu (null/nan) tren ca 3 tep")

Da xu ly gia thieu (null/nan) tren ca 3 tep


In [5]:
# # Tiền xử lý văn bản tiếng việt
import re
import unicodedata

def clean_vietnamese_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for df in [train_df, val_df, test_df]:
    df['clean_comment'] = df['Comment'].apply(clean_vietnamese_text)
    df['char_count'] = df['clean_comment'].apply(len)
    df['word_count'] = df['clean_comment'].apply(lambda x: len(x.split()))
train_df[['Comment', 'clean_comment', 'char_count', 'word_count']].head(3)


,Comment,clean_comment,char_count,word_count
0,sản phẩm đúng quảng cáo. giao hàng hơi chậm nh...,sản phẩm đúng quảng cáo. giao hàng hơi chậm nh...,61,14
1,"Mình nhận hàng ngày hôm qua, tiki giao hàng đú...","mình nhận hàng ngày hôm qua, tiki giao hàng đú...",425,96
2,"Máy giặt quá tốt, chỉ có 5 tr được cái máy giặ...","máy giặt quá tốt, chỉ có 5 tr được cái máy giặ...",91,21


In [6]:
# Tạo đặc trưng khía cạnh mua sắm
# Danh sách các từ khóa theo khía cạnh mua sắm
aspect_keywords = {
    'aspect_delivery': ['giao hàng', 'vận chuyển', 'shipper', 'ship', 'nhanh', 'chậm', 'đóng gói'],
    'aspect_quality': ['chất lượng', 'vải', 'màu', 'đẹp', 'chất', 'bền', 'dùng tốt', 'xịn', 'xấu'],
    'aspect_price': ['giá', 'rẻ', 'đắt', 'tiền', 'đáng mua', 'sale', 'khuyến mãi']
}
def  extract_aspects(text, keywords):
    return int(any(kw in text for kw in keywords))
for df in [train_df, val_df, test_df]:
    for aspect_name, keywords in aspect_keywords.items():
        df[aspect_name] = df['Comment'].apply(lambda x: extract_aspects(x, keywords))
train_df[['aspect_delivery', 'aspect_quality', 'aspect_price']].mean()

aspect_delivery    0.381239
aspect_quality     0.366000
aspect_price       0.247588
dtype: float64